In [ ]:
# -*- coding: utf-8 -*-
"""
ResNet-18 backbone + Multi-Head (Task-IL) on CIFAR-10
Grid-Search over: batch_size, epochs, learning_rate
- 5 tasks (2 classes each): [0,1],[2,3],[4,5],[6,7],[8,9]
- Train/Select ONLY Task-1 (classes 0 & 1) via 10% validation split
- Epoch-only logging (one line per epoch)
- Device-safe: works on CUDA / MPS / CPU automatically
- After selection, retrain on (train+val) with best config and save checkpoint
"""
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
import random
from typing import Dict, List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms
from copy import deepcopy
# -----------------------------
# General settings
# -----------------------------
SEED = 42
DATA_ROOT = "./data"
NUM_WORKERS = 2
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark"
os.makedirs(SAVE_DIR, exist_ok=True)

# GroupNorm settings
GN_GROUPS = 32

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Automatic device selection
# -----------------------------
def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()
print(f"[INFO] Using device: {device}")

PIN_MEM = (device.type == "cuda")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

# ----------------------------------------------------------
# Utilities for GroupNorm
# ----------------------------------------------------------
def make_gn(C: int) -> nn.GroupNorm:
    """
    Create a GroupNorm layer with a valid number of groups for C channels.
    Starts from GN_GROUPS and reduces it until it divides C.
    Falls back to 1 group if needed.
    """
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

# ----------------------------------------------------------
# ResNet-18 backbone adapted for CIFAR-10
# ----------------------------------------------------------
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                make_gn(planes * self.expansion)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        block = BasicBlock
        num_blocks = [2, 2, 2, 2]
        self.expansion = block.expansion
        self.nf = nf
        self.in_planes = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)

        self.layer1 = self._make_layer(block, nf,     num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, nf * 2, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, nf * 4, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, nf * 8, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        in_planes = self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = torch.nn.functional.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self) -> int:
        return self.nf * 8 * self.expansion
# -----------------------------------------
# Multi-head model
# -----------------------------------------
class MultiHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()

    def add_head(self, task_name: str, num_classes: int):
        if task_name in self.heads:
            raise ValueError(f"Head '{task_name}' already exists.")
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[task_name] = head

    def forward(self, x: torch.Tensor, task_name: str) -> torch.Tensor:
        if task_name not in self.heads:
            raise ValueError(f"Head '{task_name}' does not exist.")
        feat = self.backbone(x)
        logits = self.heads[task_name](feat)
        return logits

# ------------------------------------------------------
# Split CIFAR-10 into 5 tasks
# ------------------------------------------------------
def build_tasks() -> Dict[str, List[int]]:
    return {
        "task1": [0, 1],
        "task2": [2, 3],
        "task3": [4, 5],
        "task4": [6, 7],
        "task5": [8, 9],
    }

def filter_indices_by_classes(dataset, classes: List[int]) -> List[int]:
    t = dataset.targets if hasattr(dataset, "targets") else [dataset[i][1] for i in range(len(dataset))]
    return [i for i, y in enumerate(t) if y in classes]

def remap_labels(original_targets: List[int], keep_classes: List[int]) -> List[int]:
    class_to_new = {c: i for i, c in enumerate(sorted(keep_classes))}
    return [class_to_new[y] for y in original_targets]

class RelabeledSubset(Subset):
    def __init__(self, dataset, indices: List[int], keep_classes: List[int]):
        super().__init__(dataset, indices)

        if hasattr(dataset, "targets"):
            original_targets = [dataset.targets[i] for i in indices]
        else:
            original_targets = [dataset[i][1] for i in indices]

        self.new_targets = remap_labels(original_targets, keep_classes)

    def __getitem__(self, idx):
        x, _ = super().__getitem__(idx)
        y_new = self.new_targets[idx]
        return x, y_new

    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]

# -----------------------------------------
# CIFAR-10 datasets
# -----------------------------------------
def get_cifar10_datasets():
    tf_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                             std=(0.2470, 0.2435, 0.2616)),
    ])
    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                             std=(0.2470, 0.2435, 0.2616)),
    ])
    train_set = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tf_train)
    test_set  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tf_test)
    return train_set, test_set

def make_task_subset(dataset, task_classes: List[int]) -> RelabeledSubset:
    idx = filter_indices_by_classes(dataset, task_classes)
    return RelabeledSubset(dataset, idx, task_classes)

def make_loader(ds, batch_size: int, shuffle: bool) -> DataLoader:
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEM)

# -----------------------------------------
# Evaluation
# -----------------------------------------
@torch.no_grad()
def evaluate(model: MultiHeadNet, task_name: str, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x, task_name)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)
    return correct / max(1, total)

# -----------------------------------------
# Train a single configuration with epoch-level logging
# -----------------------------------------
def train_one_config(model: MultiHeadNet,
                     task_name: str,
                     train_loader: DataLoader,
                     val_loader: DataLoader,
                     epochs: int,
                     lr: float):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    best_val = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(x, task_name)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_samples += x.size(0)

        avg_loss = total_loss / max(1, total_samples)
        train_acc = total_correct / max(1, total_samples)
        val_acc = evaluate(model, task_name, val_loader)
        best_val = max(best_val, val_acc)

        print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% "
              f"(Best Val: {best_val*100:.2f}%)")

    return best_val

# -----------------------------------------
# Grid Search
# -----------------------------------------
def grid_search_task1():
    tasks = build_tasks()
    task1_classes = tasks["task1"]

    train_set, test_set = get_cifar10_datasets()
    task1_train = make_task_subset(train_set, task1_classes)
    task1_test  = make_task_subset(test_set,  task1_classes)

    # split train into train/val (90%/10%)
    val_len = max(1, int(0.1 * len(task1_train)))
    train_len = len(task1_train) - val_len
    task1_train_split, task1_val_split = random_split(
        task1_train, [train_len, val_len],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Build the model with five task-specific heads
    backbone = ResNet18Backbone(nf=64)
    model = MultiHeadNet(backbone=backbone)
    for t in ["task1", "task2", "task3", "task4", "task5"]:
        model.add_head(t, num_classes=2)

    # Move the model to the selected device
    model.to(device)

    # Hyperparameter search space
    batch_sizes = [64,128,256]
    epochs_list = [60]
    lrs = [0.1]

    best_cfg = None
    best_val = -1.0

    # Save the initial model state for reinitialization before each configuration
    init_state = model.state_dict()

    for bs in batch_sizes:
        train_loader = make_loader(task1_train_split, batch_size=bs, shuffle=True)
        val_loader   = make_loader(task1_val_split,   batch_size=bs, shuffle=False)

        for epochs in epochs_list:
            for lr in lrs:
                print(f"\n=== Trying config: batch_size={bs}, epochs={epochs}, lr={lr} ===")
                # Reset the model before training each configuration
                model.load_state_dict(init_state)
                model.to(device)

                val_acc = train_one_config(model, "task1", train_loader, val_loader,
                                           epochs=epochs, lr=lr)

                if val_acc > best_val:
                    best_val = val_acc
                    best_cfg = {"batch_size": bs, "epochs": epochs, "lr": lr}

    print("\nGrid-Search DONE.")
    print(f"Best Val Acc: {best_val*100:.2f}% with {best_cfg}")

    # ---------------------------------------
    # Retrain on the full training set (train + validation) using the best configuration
    # ---------------------------------------
    model.load_state_dict(init_state)
    model.to(device)

    #Build data loaders using the best batch size
    bs = best_cfg["batch_size"]
    epochs = best_cfg["epochs"]
    lr = best_cfg["lr"]

    full_train = ConcatDataset([task1_train_split, task1_val_split])
    full_train_loader = make_loader(full_train, batch_size=bs, shuffle=True)
    test_loader       = make_loader(task1_test,  batch_size=bs, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

    # Track the best test accuracy and corresponding model state
    best_test = -1.0
    best_epoch = -1
    best_retrain_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for x, y in full_train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(x, "task1")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_samples += x.size(0)

        avg_loss = total_loss / max(1, total_samples)
        train_acc = total_correct / max(1, total_samples)
        test_acc = evaluate(model, "task1", test_loader)

        # Track the best-performing model
        if test_acc > best_test:
            best_test = test_acc
            best_epoch = epoch
            best_retrain_state = deepcopy(model.state_dict())

        print(f"[Final] Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

    # Save the best checkpoint for subsequent fine-tuning
    if best_retrain_state is None:
        best_retrain_state = model.state_dict()

    best_test_path = os.path.join(SAVE_DIR, "Gtask1_best_test_for_finetune.pth")
    torch.save({
        "model_state": best_retrain_state,
        "best_config": best_cfg,
        "best_test_acc": best_test,
        "best_test_epoch": best_epoch,
        "trained_head": "task1",
        "all_heads": list(model.heads.keys())
    }, best_test_path)
    print(f"Saved BEST-TEST checkpoint -> {best_test_path} (epoch {best_epoch}, acc {best_test*100:.2f}%)")


# -----------------------------------------
# Main
# -----------------------------------------
if __name__ == "__main__":
    grid_search_task1()


Mounted at /content/drive
[INFO] Using device: cuda


100%|██████████| 170M/170M [32:21<00:00, 87.8kB/s]



=== Trying config: batch_size=64, epochs=60, lr=0.1 ===
Epoch 001 | Train Loss: 1.8322 | Train Acc: 52.33% | Val Acc: 52.30% (Best Val: 52.30%)
Epoch 002 | Train Loss: 0.6589 | Train Acc: 59.97% | Val Acc: 67.90% (Best Val: 67.90%)
Epoch 003 | Train Loss: 0.5483 | Train Acc: 72.88% | Val Acc: 75.10% (Best Val: 75.10%)
Epoch 004 | Train Loss: 0.5194 | Train Acc: 75.38% | Val Acc: 78.80% (Best Val: 78.80%)
Epoch 005 | Train Loss: 0.4747 | Train Acc: 78.18% | Val Acc: 81.80% (Best Val: 81.80%)
Epoch 006 | Train Loss: 0.4697 | Train Acc: 78.79% | Val Acc: 82.40% (Best Val: 82.40%)
Epoch 007 | Train Loss: 0.3867 | Train Acc: 82.92% | Val Acc: 83.90% (Best Val: 83.90%)
Epoch 008 | Train Loss: 0.3575 | Train Acc: 84.71% | Val Acc: 86.10% (Best Val: 86.10%)
Epoch 009 | Train Loss: 0.3171 | Train Acc: 86.59% | Val Acc: 87.10% (Best Val: 87.10%)
Epoch 010 | Train Loss: 0.3031 | Train Acc: 87.50% | Val Acc: 88.20% (Best Val: 88.20%)
Epoch 011 | Train Loss: 0.2940 | Train Acc: 87.76% | Val Acc: 8